# 01 — Tái lập CausalGym (RQ0, cổng G1)

**Mục tiêu:** tái lập bảng odds và selectivity của CausalGym (Arora, Jurafsky, Potts, ACL 2024) trên Pythia ở **fp32**, cho các tác vụ trong `TASKS`, với đủ phương pháp của bài gốc: DAS, probe (2 mức regularisation), diff-in-means, PCA, k-means, LDA, hướng ngẫu nhiên, và vanilla (thay cả vector, Phụ lục E của bài).

**Cách làm khớp repo gốc** (`aryamanarora/causalgym`, commit `0f3129f`; sổ dự án §8):
- can thiệp 1 chiều ở residual stream sau mỗi layer, tại token cuối của mỗi vùng;
- DAS: 1 epoch, batch 4, Adam, lr 5·10⁻³, warmup tuyến tính 10%, loss cross-entropy về nhãn src;
- probe: LogisticRegression saga, L2, có bias, C theo cỡ model (410m: 1e-4 và 1e-5; 1b: 1e-5 và 1e-6);
- odds = [logit(y_src) − logit(y_base)] sau can thiệp trừ trước can thiệp;
- tác vụ đối chứng `dog-give`: thay nhãn theo loại (`types[0]` → " dog", loại còn lại → " give"); selectivity = odds gốc − odds đối chứng;
- bảng tổng hợp: ở mỗi layer lấy max theo vùng, rồi trung bình qua layer (`plot.py`).

**Chỗ cố ý khác repo gốc** (ghi trong sổ, D-13):
- `POS_MODE = "correct"`: lấy biểu diễn source ở **vị trí của source**. Repo gốc truyền vị trí của base cho cả source (`train.py:62`, `eval.py:31`, `data.py:140`), nên khi base và source khác độ dài token thì lấy sai chỗ. `POS_MODE = "repo"` tái hiện đúng cách của repo gốc để so.
- mốc FP luôn là fp32 (repo gốc chạy 1b ở bfloat16).
- tách token từng vùng rồi nối (repo gốc tách cả câu rồi căn vùng); ô kiểm dữ liệu đếm số câu lệch.

**Tối ưu tốc độ (không đổi kết quả):** biểu diễn source và base ở mọi layer × vùng được tính một lần cho mỗi tác vụ (model đóng băng), nên mỗi bước học DAS chỉ cần một lượt forward có can thiệp.

**Cách chạy mặc định:** sửa `configs/kaggle_run.yaml` → commit → push → Kaggle runner → *Save Version*. Kiểm code không cần GPU: `TOY_TEST = True`.

In [1]:
# parameters — papermill ghi đè các giá trị này từ configs/kaggle_run.yaml
MODEL = "EleutherAI/pythia-410m"
REVISION = "main"
TASKS = ["agr_sv_num_subj-relc", "npi_any_subj-relc", "filler_gap_subj", "cleft"]
TRAIN_SPLIT = "train"
EVAL_SPLIT = "test"         # repo gốc đánh giá trên tập seed 1, ứng với split test của HF; không có thì dùng validation
MAX_TRAIN = 400
MAX_EVAL = 100
METHODS = ["vanilla", "das", "probe0", "probe1", "mean", "pca", "kmeans", "lda", "random"]
PROBE_C = []                # rỗng: lấy theo model như repo gốc; hoặc hai giá trị C cho probe0, probe1
CONTROL = True              # chạy thêm tác vụ đối chứng dog-give để tính selectivity
POS_MODE = "correct"        # "correct": source lấy ở vị trí của source; "repo": như code gốc
LAYERS = "all"              # "all" hoặc list layer
REGIONS = "all"             # "all": từ vùng khác nhau đầu tiên tới vùng cuối; hoặc list
SAVE_PAIRS = True           # lưu odds từng cặp của das và vanilla (tác vụ gốc)
SAVE_DIRS = True            # lưu hướng DAS của tác vụ gốc ra file npz
BS = 4
EVAL_BS = 32
CAP_BS = 32                 # batch khi tính trước biểu diễn
EPOCHS = 1
LR = 5e-3
SEED = 0
MAX_HOURS = 10.0            # quá số giờ này thì dừng sau tác vụ đang chạy
DTYPE = "float32"
OUT_DIR = "outputs/01_local"
REPO_DIR = "."
TOY_TEST = False            # True: model và dữ liệu giả trên CPU, chỉ để kiểm code

In [2]:
# Parameters
MODEL = "EleutherAI/pythia-1b"
TASKS = ["agr_sv_num_subj-relc", "npi_any_subj-relc", "filler_gap_subj", "cleft"]
MAX_HOURS = 4.5
OUT_DIR = "/kaggle/working/outputs/20261113_0846_g1/g1_1b"
REPO_DIR = "/tmp/repo"


## 1. Môi trường

In phiên bản thư viện và GPU (QT-26). Kiểm tham số hợp lệ trước khi tải model.

In [3]:
import gc, json, math, os, random, time, warnings
from collections import Counter
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn.functional as F
import transformers

T_START = time.time()
DEV = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUT_DIR, exist_ok=True)
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device", DEV,
      "|", torch.cuda.get_device_name(0) if DEV == "cuda" else "CPU")
if DEV == "cpu" and not TOY_TEST:
    print("CẢNH BÁO: không thấy GPU. Kiểm tra Settings → Accelerator trên Kaggle.")

KNOWN_METHODS = {"vanilla", "das", "probe0", "probe1", "mean", "pca", "kmeans", "lda", "random"}
bad = [m for m in METHODS if m not in KNOWN_METHODS]
if bad:
    raise ValueError(f"Phương pháp không hỗ trợ: {bad}")
if POS_MODE not in ("correct", "repo"):
    raise ValueError("POS_MODE phải là 'correct' hoặc 'repo'")
MANIPS = ["none", "dog-give"] if CONTROL else ["none"]

torch 2.11.0+cu130 | transformers 5.20.2 | device cuda | Tesla T4


## 2. Hàm tiện ích

- `Logger`: ghi mỗi dòng kết quả vào JSONL ngay khi có (QT-23). Dòng của từng ô không in ra màn hình để `executed.ipynb` gọn; mỗi ô in một dòng tóm tắt.
- `kendall_tau`: τ_b, dùng để so thứ tự phương pháp với bài gốc.

In [4]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

class Logger:
    def __init__(self, path):
        self.path = path
    def __call__(self, rec, show=True):
        rec = dict(rec, time=time.strftime("%Y-%m-%d %H:%M:%S"))
        if show:
            print(json.dumps({k: v for k, v in rec.items() if not isinstance(v, (list, dict))}, ensure_ascii=False), flush=True)
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def kendall_tau(a, b):
    a, b = list(a), list(b)
    conc = disc = ta = tb = 0
    for i in range(len(a)):
        for j in range(i + 1, len(a)):
            s1, s2 = np.sign(a[i] - a[j]), np.sign(b[i] - b[j])
            if s1 == 0 and s2 == 0:
                continue
            if s1 == 0:
                ta += 1
            elif s2 == 0:
                tb += 1
            elif s1 == s2:
                conc += 1
            else:
                disc += 1
    den = math.sqrt((conc + disc + ta) * (conc + disc + tb))
    return (conc - disc) / den if den else float("nan")

def hours():
    return (time.time() - T_START) / 3600

log = Logger(os.path.join(OUT_DIR, "results.jsonl"))

## 3. Số liệu của bài gốc để đối chiếu

Overall odds-ratio theo tác vụ, lấy từ Phụ lục E.2 của bài (Table 11: pythia-410m; Table 13: pythia-1b), kèm accuracy của tác vụ. `probe0`/`probe1` là hai mức regularisation của bài. Tác vụ không có ở đây thì bỏ qua phần đối chiếu.

Thứ tự loại (`types`) lấy từ template của repo gốc (`data/templates/syntaxgym.json`, khoá của `labels`), dùng để gán nhãn dog-give đúng như repo.

In [5]:
PAPER = {
    "pythia-410m": {   # Table 11
        "agr_sv_num_subj-relc": dict(acc=0.97, das=4.97, probe0=4.32, probe1=4.19, mean=3.72, pca=0.29, kmeans=0.28, lda=1.88, random=0.03, vanilla=4.42),
        "npi_any_subj-relc":    dict(acc=0.95, das=8.76, probe0=4.01, probe1=4.00, mean=3.99, pca=2.28, kmeans=2.34, lda=0.15, random=0.01, vanilla=4.08),
        "filler_gap_subj":      dict(acc=0.89, das=11.84, probe0=6.53, probe1=6.44, mean=6.41, pca=4.87, kmeans=4.98, lda=0.14, random=0.01, vanilla=6.34),
        "cleft":                dict(acc=0.95, das=14.46, probe0=5.53, probe1=4.84, mean=1.78, pca=0.86, kmeans=0.92, lda=0.05, random=0.02, vanilla=5.79),
    },
    "pythia-1b": {     # Table 13
        "agr_sv_num_subj-relc": dict(acc=1.00, das=6.62, probe0=5.23, probe1=4.98, mean=4.24, pca=0.36, kmeans=0.39, lda=1.40, random=0.01, vanilla=5.51),
        "npi_any_subj-relc":    dict(acc=0.97, das=10.51, probe0=4.33, probe1=4.33, mean=4.33, pca=2.80, kmeans=2.93, lda=0.20, random=0.00, vanilla=4.41),
        "filler_gap_subj":      dict(acc=0.89, das=11.82, probe0=6.23, probe1=6.18, mean=6.17, pca=3.76, kmeans=3.87, lda=0.03, random=0.00, vanilla=6.22),
        "cleft":                dict(acc=0.97, das=15.56, probe0=5.59, probe1=3.76, mean=1.62, pca=0.25, kmeans=0.43, lda=0.16, random=0.01, vanilla=6.04),
    },
}
TYPE_ORDER = {
    "agr_sv_num_subj-relc": ["plural", "singular"],
    "npi_any_subj-relc": ["any", "some"],
    "filler_gap_subj": ["that", "wh"],
    "cleft": ["pseudo-cleft", "normal"],
}
PROBE_C_BY_MODEL = {"pythia-70m": [1e-3, 1e-3], "pythia-160m": [1e-4, 1e-5], "pythia-410m": [1e-4, 1e-5],
                    "pythia-1b": [1e-5, 1e-6], "pythia-1.4b": [1e-5, 1e-6]}
MODEL_KEY = MODEL.split("/")[-1]
PAPER_M = PAPER.get(MODEL_KEY, {})

## 4. Dữ liệu

- Mỗi câu là danh sách vùng. Tách token từng vùng rồi nối; ghi token cuối mỗi vùng (`b_end`, `s_end`).
- Nhãn dog-give gán theo loại của câu, như `data.py:323` của repo gốc.
- `POS_MODE` quyết định source lấy ở đâu: `correct` lấy ở `s_end`, `repo` lấy ở `b_end` (như repo gốc). Base luôn bị can thiệp ở `b_end`. Biểu diễn dùng để học probe/mean/PCA... lấy trên base ở `b_end` (`correct`) hoặc ở `s_end` (`repo`, như đơn vị Collect của repo gốc).
- Base và source của cùng một lô được đệm phải tới cùng độ dài, như repo gốc.

In [6]:
DS_CACHE = {}

def load_split(split):
    if split not in DS_CACHE:
        from datasets import load_dataset
        DS_CACHE[split] = load_dataset("aryaman/causalgym", split=split)
    return DS_CACHE[split]

def load_rows(task, split):
    rows = [r for r in load_split(split) if r["task"] == task]
    if not rows:
        raise ValueError(f"Không có task '{task}' trong split {split}.")
    return rows

def resolve_eval_split(split):
    try:
        load_split(split)
        return split
    except Exception as e:
        print(f"Không tải được split '{split}' ({type(e).__name__}: {e}). Dùng 'validation'.")
        load_split("validation")
        return "validation"

def encode_spans(tok, spans):
    ids, ends = [], []
    for s in spans:
        ids += tok(s, add_special_tokens=False)["input_ids"]
        ends.append(len(ids) - 1)
    return ids, ends

def first_tok(tok, s):
    t = tok(s, add_special_tokens=False)["input_ids"]
    return t[0], len(t)

def build_examples(tok, rows, types):
    dog, _ = first_tok(tok, " dog")
    give, _ = first_tok(tok, " give")
    exs, multi = [], 0
    for r in rows:
        b_ids, b_end = encode_spans(tok, r["base"])
        s_ids, s_end = encode_spans(tok, r["src"])
        yb, nb = first_tok(tok, r["base_label"])
        ys, ns = first_tok(tok, r["src_label"])
        multi += int(nb > 1) + int(ns > 1)
        if len(b_end) != len(s_end) or yb == ys:
            continue
        bt, st = r["base_type"], r["src_type"]
        diff = [i for i, (a, b) in enumerate(zip(r["base"], r["src"])) if a != b]
        b_txt, s_txt = "".join(r["base"]), "".join(r["src"])
        exs.append(dict(b_ids=b_ids, s_ids=s_ids, b_end=b_end, s_end=s_end, yb=yb, ys=ys,
                        yb_dg=dog if bt == types[0] else give, ys_dg=dog if st == types[0] else give,
                        bt=bt, st=st, diff=diff, b_txt=b_txt, s_txt=s_txt, b_spans=list(r["base"]),
                        s_spans=list(r["src"])))
    return exs, multi

def find_regions(exs):
    n_reg = Counter(len(e["b_end"]) for e in exs).most_common(1)[0][0]
    diffs = Counter(e["diff"][0] for e in exs if e["diff"])
    return n_reg, (diffs.most_common(1)[0][0] if diffs else 0)

def pad_to(seqs, L, pad_id, dev):
    ids = torch.full((len(seqs), L), pad_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
        mask[i, :len(s)] = 1
    last = torch.tensor([len(s) - 1 for s in seqs])
    return ids.to(dev), mask.to(dev), last.to(dev)

def label_keys(manip):
    return ("yb", "ys") if manip == "none" else ("yb_dg", "ys_dg")

## 5. Can thiệp 1D và các hàm đo

- `MultiCapture`: một lượt forward bắt biểu diễn ở mọi layer cần đo, tại vị trí cuối của mọi vùng.
- `Site`: hook vào output của `gpt_neox.layers[l]`. Với hướng đơn vị v: h ← h + ((h_src − h)·v) v. Với vanilla (v = None): h ← h_src.
- `train_das`: như notebook 00, nhưng dùng biểu diễn source đã tính trước.
- `eval_dir`: odds và IIA cho một hướng, dùng logit của base đã tính trước.

In [7]:
class MultiCapture:
    def __init__(self, model, layers):
        self.pos, self.out = None, {}
        self.handles = [model.gpt_neox.layers[l].register_forward_hook(self._mk(l)) for l in layers]
    def _mk(self, l):
        def hook(module, inputs, out):
            hs = out[0] if isinstance(out, tuple) else out
            idx = torch.arange(hs.shape[0], device=hs.device)[:, None]
            self.out[l] = hs[idx, self.pos].detach().float().cpu()
        return hook
    def remove(self):
        for h in self.handles:
            h.remove()

@torch.no_grad()
def capture_all(model, exs, which, pos_key, layers, regions, args, dev):
    # which: "b" hoặc "s" (câu nào chạy qua model); pos_key: "b_end" hoặc "s_end" (lấy ở vị trí nào)
    cap, outs = MultiCapture(model, layers), {l: [] for l in layers}
    try:
        for i in range(0, len(exs), args.cap_bs):
            chunk = exs[i:i + args.cap_bs]
            L = max(max(len(e["b_ids"]), len(e["s_ids"])) for e in chunk)
            ids, mask, _ = pad_to([e[which + "_ids"] for e in chunk], L, args.pad_id, dev)
            cap.pos = torch.tensor([[e[pos_key][r] for r in regions] for e in chunk], device=dev)
            model(input_ids=ids, attention_mask=mask)
            for l in layers:
                outs[l].append(cap.out[l])
    finally:
        cap.remove()
    return {l: torch.cat(outs[l]) for l in layers}

class Site:
    def __init__(self, model, layer):
        self.pos, self.src, self.v, self.on = None, None, None, False
        self.handle = model.gpt_neox.layers[layer].register_forward_hook(self._hook)
    def remove(self):
        self.handle.remove()
    def _hook(self, module, inputs, out):
        if not self.on:
            return None
        hs = out[0] if isinstance(out, tuple) else out
        idx = torch.arange(hs.shape[0], device=hs.device)
        hb = hs[idx, self.pos]
        src = self.src.to(hs.dtype)
        if self.v is None:
            val = src
        else:
            v = self.v.to(hs.dtype)
            val = hb + ((src - hb) @ v)[:, None] * v[None, :]
        new = hs.clone()
        new[idx, self.pos] = val
        return (new,) + tuple(out[1:]) if isinstance(out, tuple) else new

def last_logits(model, ids, mask, last):
    out = model(input_ids=ids, attention_mask=mask)
    return out.logits[torch.arange(ids.shape[0], device=ids.device), last].float()

def intervened_logits(model, site, exs, region, src, v, args, dev):
    ids, mask, last = pad_to([e["b_ids"] for e in exs], max(len(e["b_ids"]) for e in exs), args.pad_id, dev)
    site.pos = torch.tensor([e["b_end"][region] for e in exs], device=dev)
    site.src, site.v, site.on = src.to(dev), v, True
    try:
        return last_logits(model, ids, mask, last)
    finally:
        site.on = False

def train_das(model, layer, region, exs, src_reps, manip, seed, args, dev):
    yb_k, ys_k = label_keys(manip)
    set_seed(seed)
    d = model.config.hidden_size
    w = torch.nn.Parameter(torch.randn(d, device=dev) / math.sqrt(d))
    opt = torch.optim.Adam([w], lr=args.lr)
    order = list(range(len(exs))); random.Random(seed).shuffle(order)
    batches = [order[i:i + args.bs] for i in range(0, len(order), args.bs)] * args.epochs
    total, warm = len(batches), max(1, int(0.1 * len(batches)))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    site = Site(model, layer)
    sync(); t0 = time.time()
    try:
        for bi in batches:
            chunk = [exs[i] for i in bi]
            logits = intervened_logits(model, site, chunk, region, src_reps[bi], w / w.norm(), args, dev)
            loss = F.cross_entropy(logits, torch.tensor([e[ys_k] for e in chunk], device=dev))
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    finally:
        site.remove()
    sync()
    return (w / w.norm()).detach(), time.time() - t0

@torch.no_grad()
def base_lo(model, exs, manip, args, dev):
    yb_k, ys_k = label_keys(manip)
    lo = []
    for i in range(0, len(exs), args.eval_bs):
        chunk = exs[i:i + args.eval_bs]
        ids, mask, last = pad_to([e["b_ids"] for e in chunk], max(len(e["b_ids"]) for e in chunk), args.pad_id, dev)
        lg = last_logits(model, ids, mask, last)
        ar = torch.arange(len(chunk), device=dev)
        lo += (lg[ar, [e[ys_k] for e in chunk]] - lg[ar, [e[yb_k] for e in chunk]]).tolist()
    return np.asarray(lo)

@torch.no_grad()
def eval_dir(model, layer, region, exs, src_reps, v, lo_b, manip, args, dev):
    yb_k, ys_k = label_keys(manip)
    site, lo_i = Site(model, layer), []
    try:
        for i in range(0, len(exs), args.eval_bs):
            chunk = exs[i:i + args.eval_bs]
            lg = intervened_logits(model, site, chunk, region, src_reps[i:i + args.eval_bs], v, args, dev)
            ar = torch.arange(len(chunk), device=dev)
            lo_i += (lg[ar, [e[ys_k] for e in chunk]] - lg[ar, [e[yb_k] for e in chunk]]).tolist()
    finally:
        site.remove()
    lo_i = np.asarray(lo_i)
    per = lo_i - lo_b
    return float(per.mean()), float((lo_i > 0).mean()), per

## 6. Các phương pháp không học qua đầu ra

Học trên biểu diễn base của tập train với nhãn là loại của câu base, như `diff_methods.py` của repo gốc. Mọi hướng được chuẩn hoá về độ dài 1; dấu của hướng không ảnh hưởng can thiệp 1D. Probe và LDA ghi thêm accuracy phân loại trên tập eval.

In [8]:
def make_dir(method, X, y, Xev, yev, probe_c, seed):
    from sklearn.cluster import KMeans
    from sklearn.decomposition import PCA
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
    from sklearn.linear_model import LogisticRegression
    acc = None
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        if method == "mean":
            cls = sorted(set(y))
            vec = X[y == cls[1]].mean(0) - X[y == cls[0]].mean(0)
        elif method == "kmeans":
            km = KMeans(n_clusters=2, random_state=0, n_init=10).fit(X)
            vec = km.cluster_centers_[0] - km.cluster_centers_[1]
        elif method == "pca":
            vec = PCA(n_components=1).fit(X).components_[0]
        elif method == "lda":
            m = LinearDiscriminantAnalysis(n_components=1).fit(X, y)
            vec, acc = m.coef_[0], float(m.score(Xev, yev))
        elif method in ("probe0", "probe1"):
            m = LogisticRegression(C=probe_c[0 if method == "probe0" else 1], solver="saga", max_iter=1000,
                                   fit_intercept=True, random_state=0).fit(X, y)
            vec, acc = m.coef_[0], float(m.score(Xev, yev))
        elif method == "random":
            vec = np.random.default_rng(seed).standard_normal(X.shape[1])
        else:
            raise ValueError(method)
    vec = torch.tensor(np.asarray(vec, dtype=np.float64), dtype=torch.float32)
    n = float(vec.norm())
    return (vec / n if n > 0 else vec), acc

## 7. Nạp model và tokenizer

Nếu `TOY_TEST = True`: tokenizer, model và dữ liệu giả (không tải gì). Dữ liệu giả có cặp đảo, có vùng khác độ dài token giữa base và source để kiểm cả hai `POS_MODE`.

In [9]:
if TOY_TEST:
    from transformers import GPTNeoXConfig, GPTNeoXForCausalLM
    class ToyTok:
        pad_token_id, eos_token_id = None, 0
        def __init__(self):
            self.vocab = {}
        def __call__(self, s, add_special_tokens=False):
            ids = []
            if s.startswith("<|endoftext|>"):
                ids.append(0); s = s[len("<|endoftext|>"):]
            for w in s.split():
                self.vocab.setdefault(w, 2 + len(self.vocab) % 100)
                ids.append(self.vocab[w])
            return {"input_ids": ids or [1]}
        def convert_ids_to_tokens(self, ids):
            inv = {v: k for k, v in self.vocab.items()}
            inv[0] = "<|endoftext|>"
            return [inv.get(i, "?") for i in ids]
    def load_split(split):
        return split
    def load_rows(task, split):
        rng, out = random.Random(f"{task}-{split}"), []
        nouns = [(" man", " big men"), (" dog", " dogs"), (" cat", " old cats")]
        for _ in range(12 if split == "train" else 6):
            sg = rng.random() < 0.5
            n1, n2 = rng.choice(nouns), rng.choice(nouns)
            b = ["<|endoftext|>The", n1[0] if sg else n1[1], " near the car", " that sleeps"]
            s = ["<|endoftext|>The", n2[1] if sg else n2[0], " near the car", " that sleeps"]
            lb, ls = (" is", " are") if sg else (" are", " is")
            tb, ts = ("singular", "plural") if sg else ("plural", "singular")
            out.append(dict(base=b, src=s, base_label=lb, src_label=ls, base_type=tb, src_type=ts, task=task))
            out.append(dict(base=s, src=b, base_label=ls, src_label=lb, base_type=ts, src_type=tb, task=task))
        return out
    def load_model(name, revision, dtype, dev):
        torch.manual_seed(0)
        cfg = GPTNeoXConfig(vocab_size=120, hidden_size=64, num_hidden_layers=6, num_attention_heads=4,
                            intermediate_size=256, max_position_embeddings=64)
        m = GPTNeoXForCausalLM(cfg).to(dev, dtype).eval()
        for p in m.parameters():
            p.requires_grad_(False)
        return m
    tok = ToyTok()
    probe_c = list(PROBE_C) if PROBE_C else [1e-1, 1e-2]
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    def load_model(name, revision, dtype, dev):
        m = AutoModelForCausalLM.from_pretrained(name, revision=revision).to(dev, dtype)
        m.eval()
        for p in m.parameters():
            p.requires_grad_(False)
        assert hasattr(m, "gpt_neox"), "Notebook chỉ hỗ trợ Pythia / GPT-NeoX"
        return m
    tok = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
    probe_c = list(PROBE_C) if PROBE_C else PROBE_C_BY_MODEL.get(MODEL_KEY)
    if not probe_c:
        raise ValueError(f"Chưa có C mặc định cho {MODEL_KEY}; đặt PROBE_C trong config.")

args = SimpleNamespace(lr=LR, bs=BS, eval_bs=EVAL_BS, cap_bs=CAP_BS, epochs=EPOCHS,
                       pad_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)
dtype = getattr(torch, DTYPE)
eval_split = EVAL_SPLIT if TOY_TEST else resolve_eval_split(EVAL_SPLIT)
model = load_model(MODEL, REVISION, dtype, DEV)
L = model.config.num_hidden_layers
log(dict(exp="meta", model=MODEL, revision=REVISION, tasks=TASKS, toy=TOY_TEST, notebook="01", notebook_version=1,
         device=torch.cuda.get_device_name(0) if DEV == "cuda" else "cpu", torch=torch.__version__,
         transformers=transformers.__version__, dtype=DTYPE, n_layers=L, hidden=model.config.hidden_size,
         train_split=TRAIN_SPLIT, eval_split=eval_split, eval_split_requested=EVAL_SPLIT, methods=METHODS,
         probe_c=probe_c, control=CONTROL, pos_mode=POS_MODE, lr=LR, bs=BS, epochs=EPOCHS, seed=SEED))

config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

{"exp": "meta", "model": "EleutherAI/pythia-1b", "revision": "main", "toy": false, "notebook": "01", "notebook_version": 1, "device": "Tesla T4", "torch": "2.11.0+cu130", "transformers": "5.20.2", "dtype": "float32", "n_layers": 16, "hidden": 2048, "train_split": "train", "eval_split": "test", "eval_split_requested": "test", "control": true, "pos_mode": "correct", "lr": 0.005, "bs": 4, "epochs": 1, "seed": 0, "time": "2026-11-13 10:57:41"}


## 8. Chạy một tác vụ

Với mỗi tác vụ:
1. **Kiểm dữ liệu:** BOS, tách token theo vùng so với cả câu, câu eval trùng train (repo gốc chỉ loại câu base nên có thể > 0), tỉ lệ cặp đảo, **tỉ lệ cặp có vị trí base ≠ source** (chỉ ở các cặp này `POS_MODE` mới làm kết quả khác).
2. **Hành vi:** accuracy và logit diff của tác vụ trên cặp eval.
3. **Tính trước biểu diễn** source và base ở mọi layer × vùng.
4. **Mỗi ô (layer, vùng):** học/tính hướng của từng phương pháp; đánh giá trên tác vụ gốc và dog-give. Ghi một dòng `site` cho mỗi (ô, phương pháp, tác vụ con). In một dòng tóm tắt mỗi ô.
5. **Tổng hợp kiểu bài gốc:** max theo vùng ở mỗi layer → trung bình qua layer, cho odds và selectivity. So với bài gốc nếu có số.
6. **Layer xuất hiện của từng vùng** (DAS, tác vụ gốc): layer đầu tiên mà odds ≥ 0,5 × odds lớn nhất của vùng đó (chỉ vùng có max ≥ 1).

In [10]:
def paper_metric(rows, key):
    # rows: list of dict(layer, region, value); max theo vùng ở mỗi layer, rồi trung bình qua layer
    per_layer = {}
    for r in rows:
        per_layer[r["layer"]] = max(per_layer.get(r["layer"], -1e9), r[key])
    return float(np.mean(list(per_layer.values()))) if per_layer else float("nan")

def run_task(task):
    t_task = time.time()
    types_seen = sorted({r["base_type"] for r in load_rows(task, TRAIN_SPLIT)})
    types = TYPE_ORDER.get(task)
    if types is None or set(types) != set(types_seen):
        print(f"Thứ tự loại của {task} không có sẵn hoặc không khớp {types_seen}; dùng thứ tự sắp xếp.")
        types = types_seen
    tr, m1 = build_examples(tok, load_rows(task, TRAIN_SPLIT), types)
    ev, m2 = build_examples(tok, load_rows(task, eval_split), types)
    n_reg, diff_reg = find_regions(tr)
    tr = [e for e in tr if len(e["b_end"]) == n_reg][:MAX_TRAIN]
    ev = [e for e in ev if len(e["b_end"]) == n_reg][:MAX_EVAL]
    layers = list(range(L)) if LAYERS == "all" else sorted(int(x) for x in LAYERS if int(x) < L)
    regions = list(range(diff_reg, n_reg)) if REGIONS == "all" else sorted(int(x) for x in REGIONS if diff_reg <= int(x) < n_reg)
    last_r = n_reg - 1
    sites = [(l, r) for l in layers for r in regions if not (l == L - 1 and r != last_r)]

    # 1. kiểm dữ liệu
    allx = tr + ev
    tok_mis = int(sum(tok("".join(e[k + "_spans"]), add_special_tokens=False)["input_ids"] != e[k + "_ids"]
                      for e in allx for k in ("b", "s")))
    tr_txt = {e["b_txt"] for e in tr} | {e["s_txt"] for e in tr}
    ev_txt = {e["b_txt"] for e in ev} | {e["s_txt"] for e in ev}
    pair_set = {(e["b_txt"], e["s_txt"]) for e in ev}
    pos_mis = [any(e["b_end"][r] != e["s_end"][r] for r in regions) for e in ev]
    pos_mis_reg = {str(r): float(np.mean([e["b_end"][r] != e["s_end"][r] for e in ev])) for r in regions}
    e0 = ev[0]
    log(dict(exp="data_check", task=task, n_train=len(tr), n_eval=len(ev), n_regions=n_reg, diff_region=diff_reg,
             regions=regions, n_sites=len(sites), types=types, multi_token_labels=m1 + m2,
             first_token_is_eos=float(np.mean([e["b_ids"][0] == tok.eos_token_id for e in allx])),
             tok_mismatch=tok_mis, n_checked=2 * len(allx), eval_sentences_in_train=len(ev_txt & tr_txt),
             n_eval_sentences=len(ev_txt), eval_swap_frac=float(np.mean([(e["s_txt"], e["b_txt"]) in pair_set for e in ev])),
             pos_mismatch_frac=float(np.mean(pos_mis)), pos_mismatch_by_region=pos_mis_reg,
             example_spans=e0["b_spans"], example_src_spans=e0["s_spans"]))

    # 2. hành vi
    lo_b = {m: base_lo(model, ev, m, args, DEV) for m in MANIPS}
    acc = float((lo_b["none"] < 0).mean())
    log(dict(exp="behavior", task=task, acc=acc, logit_diff=float((-lo_b["none"]).mean()), n_pairs=len(ev),
             paper_acc=PAPER_M.get(task, {}).get("acc")))

    # 3. tính trước biểu diễn
    src_key, col_key = ("s_end", "b_end") if POS_MODE == "correct" else ("b_end", "s_end")
    sync(); t0 = time.time()
    TR_src = capture_all(model, tr, "s", src_key, layers, regions, args, DEV)
    EV_src = capture_all(model, ev, "s", src_key, layers, regions, args, DEV)
    need_base = any(m not in ("das", "vanilla") for m in METHODS)
    TR_base = capture_all(model, tr, "b", col_key, layers, regions, args, DEV) if need_base else None
    EV_base = capture_all(model, ev, "b", col_key, layers, regions, args, DEV) if need_base else None
    sync()
    print(f"[{task}] tính trước biểu diễn: {time.time() - t0:.1f} s; {len(sites)} ô; {len(tr)} train, {len(ev)} eval")
    y_tr = np.asarray([e["bt"] for e in tr]); y_ev = np.asarray([e["bt"] for e in ev])

    # 4. từng ô
    rows, das_dirs, t_das = [], {}, []
    for (l, r) in sites:
        ri = regions.index(r)
        t_site = time.time()
        dirs, accs = {}, {}
        if need_base:
            Xtr, Xev = TR_base[l][:, ri].numpy(), EV_base[l][:, ri].numpy()
            for m in METHODS:
                if m not in ("das", "vanilla"):
                    dirs[m], accs[m] = make_dir(m, Xtr, y_tr, Xev, y_ev, probe_c, SEED * 1000003 + l * 101 + r)
        src_tr, src_ev = TR_src[l][:, ri], EV_src[l][:, ri]
        res = {}
        for manip in MANIPS:
            for m in METHODS:
                tt = None
                if m == "das":
                    v, tt = train_das(model, l, r, tr, src_tr, manip, SEED, args, DEV)
                    t_das.append(tt)
                    if manip == "none":
                        das_dirs[(l, r)] = v.cpu()
                elif m == "vanilla":
                    v = None
                else:
                    v = dirs[m].to(DEV)
                odds, iia, per = eval_dir(model, l, r, ev, src_ev, v, lo_b[manip], manip, args, DEV)
                res[(m, manip)] = odds
                row = dict(exp="site", task=task, layer=l, region=r, method=m, manip=manip, odds=odds, iia=iia,
                           clf_acc=accs.get(m), t_train_s=round(tt, 2) if tt else None)
                log(row, show=False)
                rows.append(row)
                if SAVE_PAIRS and manip == "none" and m in ("das", "vanilla"):
                    log(dict(exp="pairs", task=task, layer=l, region=r, method=m, odds=np.round(per, 4).tolist()), show=False)
        summ = " ".join(f"{m} {res[(m, 'none')]:.2f}" for m in METHODS)
        print(f"[{task}] L{l:>2} r{r}: {summ} | {time.time() - t_site:.1f} s", flush=True)

    # 5. tổng hợp kiểu bài gốc
    summary = {}
    for m in METHODS:
        none = {(x["layer"], x["region"]): x["odds"] for x in rows if x["method"] == m and x["manip"] == "none"}
        o_rows = [dict(layer=k[0], region=k[1], v=v) for k, v in none.items()]
        s = dict(odds=paper_metric(o_rows, "v"))
        if CONTROL:
            dg = {(x["layer"], x["region"]): x["odds"] for x in rows if x["method"] == m and x["manip"] == "dog-give"}
            s["selectivity"] = paper_metric([dict(layer=k[0], region=k[1], v=none[k] - dg[k]) for k in none], "v")
        p = PAPER_M.get(task, {}).get(m)
        s["paper_odds"] = p
        s["ratio_to_paper"] = (s["odds"] / p) if p else None
        summary[m] = s
    log(dict(exp="task_summary", task=task, acc=acc, paper_acc=PAPER_M.get(task, {}).get("acc"),
             comparable_to_paper=(LAYERS == "all" and REGIONS == "all"), summary=summary,
             t_das_median_s=float(np.median(t_das)) if t_das else None, minutes=round((time.time() - t_task) / 60, 1)))

    # 6. layer xuất hiện của từng vùng (DAS, tác vụ gốc)
    arrival = {}
    if "das" in METHODS:
        for r in regions:
            col = {x["layer"]: x["odds"] for x in rows if x["method"] == "das" and x["manip"] == "none" and x["region"] == r}
            if col and max(col.values()) >= 1.0:
                top = max(col.values())
                arrival[str(r)] = min(l for l, v in col.items() if v >= 0.5 * top)
        log(dict(exp="arrival", task=task, arrival_layer=arrival, diff_region=diff_reg, last_region=last_r))

    # bảng odds DAS và vanilla dạng chữ
    for m in [x for x in ("das", "vanilla") if x in METHODS]:
        print(f"\n[{task}] odds {m} (hàng: layer, cột: vùng)")
        print("layer |" + "".join(f"{'r' + str(r):>7}" for r in regions))
        grid = {(x["layer"], x["region"]): x["odds"] for x in rows if x["method"] == m and x["manip"] == "none"}
        for l in layers:
            print(f"{l:>5} |" + "".join(f"{grid[(l, r)]:7.2f}" if (l, r) in grid else f"{'.':>7}" for r in regions))

    import pandas as pd
    pd.DataFrame(rows).drop(columns=["exp"]).to_csv(os.path.join(OUT_DIR, f"map_{task}.csv"), index=False)
    if SAVE_DIRS and das_dirs:
        keys = sorted(das_dirs)
        np.savez_compressed(os.path.join(OUT_DIR, f"das_dirs_{task}.npz"),
                            sites=np.asarray(keys), dirs=torch.stack([das_dirs[k] for k in keys]).numpy().astype(np.float32))
    del TR_src, EV_src, TR_base, EV_base
    gc.collect()
    if DEV == "cuda":
        torch.cuda.empty_cache()
    return dict(task=task, acc=acc, summary=summary, arrival=arrival, diff_region=diff_reg, last_region=last_r)

## 9. Chạy mọi tác vụ

Mỗi tác vụ ghi kết quả ngay khi xong. Nếu tổng thời gian vượt `MAX_HOURS`, dừng trước tác vụ kế tiếp và ghi lại tác vụ bị bỏ (chạy tiếp ở version sau bằng cách đặt `TASKS`).

In [11]:
done, skipped = [], []
for task in TASKS:
    if hours() > MAX_HOURS:
        skipped.append(task)
        continue
    done.append(run_task(task))
if skipped:
    log(dict(exp="skipped_tasks", tasks=skipped, hours=round(hours(), 2)))
    print("Bỏ vì quá MAX_HOURS:", skipped)

{"exp": "data_check", "task": "agr_sv_num_subj-relc", "n_train": 400, "n_eval": 100, "n_regions": 6, "diff_region": 1, "n_sites": 76, "multi_token_labels": 0, "first_token_is_eos": 1.0, "tok_mismatch": 0, "n_checked": 1000, "eval_sentences_in_train": 4, "n_eval_sentences": 99, "eval_swap_frac": 1.0, "pos_mismatch_frac": 0.16, "time": "2026-11-13 10:57:43"}


{"exp": "behavior", "task": "agr_sv_num_subj-relc", "acc": 1.0, "logit_diff": 4.380243830680847, "n_pairs": 100, "paper_acc": 1.0, "time": "2026-11-13 10:57:44"}


[agr_sv_num_subj-relc] tính trước biểu diễn: 5.1 s; 76 ô; 400 train, 100 eval


[agr_sv_num_subj-relc] L 0 r1: vanilla 8.58 das 8.20 probe0 7.93 probe1 7.91 mean 7.90 pca 0.03 kmeans -0.03 lda 0.03 random 0.00 | 21.3 s


[agr_sv_num_subj-relc] L 0 r2: vanilla 0.20 das 0.47 probe0 0.06 probe1 0.06 mean 0.06 pca 0.02 kmeans 0.05 lda 0.00 random -0.00 | 24.0 s


[agr_sv_num_subj-relc] L 0 r3: vanilla 0.04 das 0.20 probe0 0.01 probe1 0.01 mean 0.01 pca -0.00 kmeans -0.00 lda 0.00 random -0.00 | 23.2 s


... (348 outputs left out)

[cleft] L15 r3: vanilla 6.42 das 26.81 probe0 6.62 probe1 6.51 mean 6.40 pca 5.96 kmeans 6.40 lda 0.73 random 0.01 | 39.1 s


{"exp": "task_summary", "task": "cleft", "acc": 0.97, "paper_acc": 0.97, "comparable_to_paper": true, "t_das_median_s": 4.423781156539917, "minutes": 12.9, "time": "2026-11-13 13:49:48"}


{"exp": "arrival", "task": "cleft", "diff_region": 2, "last_region": 3, "time": "2026-11-13 13:49:48"}



[cleft] odds das (hàng: layer, cột: vùng)
layer |     r2     r3
    0 |  15.47   1.03
    1 |  14.52   8.40
    2 |  15.41   9.37
    3 |  16.58  13.49
    4 |  12.48  14.27
    5 |  12.28  14.50
    6 |  10.41  14.38
    7 |   7.42  13.25
    8 |   6.38  12.98
    9 |   6.30  13.40
   10 |   5.41  14.06
   11 |   5.20  15.20
   12 |   5.53  16.16
   13 |   3.98  17.51
   14 |   3.14  19.69
   15 |      .  26.81

[cleft] odds vanilla (hàng: layer, cột: vùng)
layer |     r2     r3
    0 |   6.34   0.09
    1 |   5.96   0.39
    2 |   5.57   0.78
    3 |   0.53   5.83
    4 |   0.84   5.52
    5 |   0.79   5.57
    6 |   0.11   6.28
    7 |   0.21   6.18
    8 |   0.12   6.27
    9 |   0.19   6.21
   10 |   0.04   6.36
   11 |   0.05   6.36
   12 |   0.01   6.40
   13 |  -0.07   6.48
   14 |   0.05   6.36
   15 |      .   6.42


## 10. Tóm tắt và đối chiếu với bài gốc (tiêu chí G1, sổ dự án D-13)

In ba nhóm số, **quyết định G1 do nhóm** theo tiêu chí đã chốt trong sổ:
1. **Thứ tự phương pháp:** odds trung bình qua các tác vụ đã chạy, so với trung bình của bài gốc trên cùng các tác vụ: Kendall τ_b và DAS có đứng đầu không.
2. **Thứ tự xuất hiện vị trí:** ở mỗi tác vụ, layer xuất hiện của vùng khác nhau đầu tiên có ≤ layer xuất hiện của vùng cuối không.
3. **Cùng bậc độ lớn:** tỉ lệ odds của mình / bài gốc cho DAS, probe (lấy probe tốt hơn), diff-in-means, vanilla; đếm số cặp (tác vụ, phương pháp) nằm trong [0,5; 2].

Ghi `summary_methods.csv` và `g1_check.json`.

In [12]:
import pandas as pd

tab = []
for d in done:
    for m, s in d["summary"].items():
        tab.append(dict(task=d["task"], method=m, acc=d["acc"], **s))
tab = pd.DataFrame(tab)
if len(tab):
    tab.to_csv(os.path.join(OUT_DIR, "summary_methods.csv"), index=False)
    pd.set_option("display.width", 200)
    print(tab.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

check = dict(model=MODEL_KEY, tasks=[d["task"] for d in done], pos_mode=POS_MODE, eval_split=eval_split)
with_paper = [d for d in done if d["task"] in PAPER_M]
if with_paper:
    ms = [m for m in METHODS if all(m in PAPER_M[d["task"]] for d in with_paper)]
    ours = [float(np.mean([d["summary"][m]["odds"] for d in with_paper])) for m in ms]
    paper = [float(np.mean([PAPER_M[d["task"]][m] for d in with_paper])) for m in ms]
    check["order"] = dict(methods=ms, ours=ours, paper=paper, kendall_tau=kendall_tau(ours, paper),
                          das_top=("das" in ms and ms[int(np.argmax(ours))] == "das"))
    mags = []
    for d in with_paper:
        for m in ("das", "probe", "mean", "vanilla"):
            if m == "probe":
                if not {"probe0", "probe1"} <= set(d["summary"]):
                    continue
                o = max(d["summary"]["probe0"]["odds"], d["summary"]["probe1"]["odds"])
                p = max(PAPER_M[d["task"]]["probe0"], PAPER_M[d["task"]]["probe1"])
            elif m in d["summary"]:
                o, p = d["summary"][m]["odds"], PAPER_M[d["task"]][m]
            else:
                continue
            mags.append(dict(task=d["task"], method=m, ours=o, paper=p, ratio=o / p if p else None,
                             within=bool(p and 0.5 <= o / p <= 2.0)))
    check["magnitude"] = dict(pairs=mags, n_within=sum(x["within"] for x in mags), n=len(mags))
    check["accuracy"] = [dict(task=d["task"], ours=d["acc"], paper=PAPER_M[d["task"]]["acc"],
                              abs_diff=abs(d["acc"] - PAPER_M[d["task"]]["acc"])) for d in with_paper]
check["position"] = [dict(task=d["task"], diff_region=d["diff_region"], last_region=d["last_region"],
                          arrival_diff=d["arrival"].get(str(d["diff_region"])),
                          arrival_last=d["arrival"].get(str(d["last_region"])),
                          ok=(d["arrival"].get(str(d["diff_region"])) is not None and d["arrival"].get(str(d["last_region"])) is not None
                              and d["arrival"][str(d["diff_region"])] <= d["arrival"][str(d["last_region"])]))
                     for d in done]
check["hours"] = round(hours(), 2)
log(dict(exp="g1_check", **check), show=False)
with open(os.path.join(OUT_DIR, "g1_check.json"), "w", encoding="utf-8") as f:
    json.dump(check, f, ensure_ascii=False, indent=1)

print("\n=== Đối chiếu G1 (quyết định do nhóm theo D-13) ===")
if "order" in check:
    o = check["order"]
    print("1. Thứ tự phương pháp: tau =", f"{o['kendall_tau']:.2f}", "| DAS đứng đầu:", o["das_top"])
    for m, a, b in zip(o["methods"], o["ours"], o["paper"]):
        print(f"   {m:>8}: mình {a:6.2f} | bài gốc {b:6.2f}")
for p in check["position"]:
    print(f"2. {p['task']}: layer xuất hiện vùng {p['diff_region']} = {p['arrival_diff']}, vùng cuối {p['last_region']} = {p['arrival_last']} -> {'đạt' if p['ok'] else 'không đạt'}")
if "magnitude" in check:
    g = check["magnitude"]
    print(f"3. Cùng bậc: {g['n_within']}/{g['n']} cặp trong [0,5; 2]")
    for x in g["pairs"]:
        print(f"   {x['task']:>22} {x['method']:>8}: mình {x['ours']:6.2f} | bài gốc {x['paper']:6.2f} | tỉ lệ {x['ratio']:.2f}")
    for a in check["accuracy"]:
        print(f"   accuracy {a['task']}: mình {a['ours']:.2f} | bài gốc {a['paper']:.2f}")
print("\nXong sau", round(hours(), 2), "giờ. Kết quả trong", OUT_DIR)

                task  method  acc  odds  selectivity  paper_odds  ratio_to_paper
agr_sv_num_subj-relc vanilla 1.00  5.84         7.79        5.51            1.06
agr_sv_num_subj-relc     das 1.00  7.05         4.43        6.62            1.06
agr_sv_num_subj-relc  probe0 1.00  5.58         7.46        5.23            1.07
agr_sv_num_subj-relc  probe1 1.00  5.57         7.45        4.98            1.12
agr_sv_num_subj-relc    mean 1.00  5.57         7.45        4.24            1.31
agr_sv_num_subj-relc     pca 1.00  2.66         3.52        0.36            7.39
agr_sv_num_subj-relc  kmeans 1.00  2.82         3.72        0.39            7.23
agr_sv_num_subj-relc     lda 1.00  0.26         0.36        1.40            0.19
agr_sv_num_subj-relc  random 1.00  0.00         0.00        0.01            0.11
   npi_any_subj-relc vanilla 0.97  4.41         4.82        4.41            1.00
   npi_any_subj-relc     das 0.97  9.76         3.17       10.51            0.93
   npi_any_subj-relc  probe0